objectif >>> preparation pour encodage 
 1-- Sélectionner les colonnes utiles
 2-- nettoyer les Nan 
 3-- Standardiser les features numérique avec Standard Scaler 

In [168]:
import pandas as pd 

In [169]:
#import des données

df=pd.read_csv("dataset_C_Filtre_1.csv")
df.head(1)
df.index


RangeIndex(start=0, stop=4333, step=1)

In [170]:
df_clean["Year"]

0       2009
1       2007
2       2015
3       2012
4       2012
        ... 
4328    2008
4329    2016
4330    2012
4331    2014
4332    2005
Name: Year, Length: 4333, dtype: int64

In [171]:
df["popularity"]

0       31.5683
1       14.9455
2        9.2540
3       15.7277
4       10.2175
         ...   
4328     1.4851
4329     0.1962
4330     2.4174
4331     1.1726
4332     0.5930
Name: popularity, Length: 4333, dtype: float64

In [172]:
df["genre_principal"].value_counts().sort_values(ascending=False)


genre_principal
Drame              1034
Comédie             949
Action              619
Aventure            305
Horreur             285
Thriller            185
Crime               185
Animation           122
Science-Fiction     113
Fantastique         111
Romance             108
Familial            104
Mystère              44
Musique              39
Documentaire         35
Western              34
Guerre               31
Histoire             28
Name: count, dtype: int64

In [173]:
print("colonnes :", df.shape)
print("\n--- Colonnes et types ---")
df.info()

colonnes : (4333, 50)

--- Colonnes et types ---
<class 'pandas.DataFrame'>
RangeIndex: 4333 entries, 0 to 4332
Data columns (total 50 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Title                       4333 non-null   str    
 1   title                       4333 non-null   str    
 2   original_title              4333 non-null   str    
 3   Genre                       4333 non-null   str    
 4   genres                      4333 non-null   str    
 5   Runtime                     4332 non-null   float64
 6   runtime                     4333 non-null   float64
 7   Director                    4333 non-null   str    
 8   Writer_1                    4321 non-null   str    
 9   Actors_1                    4333 non-null   str    
 10  Actors_2                    4332 non-null   str    
 11  Actors_3                    4329 non-null   str    
 12  Year                        4333 non-null   int64  


In [174]:
#Vérifications des données NULL 
df_clean.isnull().sum().sort_values(ascending=False)

genre_3            1896
genre_2             548
overview            181
Actors_3              4
genre_principal       2
genre_1               2
poster_path           1
Actors_2              1
title                 0
Director              0
Actors_1              0
Year                  0
runtime               0
vote_average          0
dtype: int64

In [175]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['Title', 'title', 'original_title', 'Genre', 'genres', 'Runtime',
       'runtime', 'Director', 'Writer_1', 'Actors_1', 'Actors_2', 'Actors_3',
       'Year', 'release_date', 'Oscars_Gagnés', 'Oscars_nommés',
       'Recompenses_globales', 'Ratings', 'imdbRating', 'imdbVotes',
       'vote_average', 'vote_count', 'popularity', 'Rated', 'Type', 'Plot',
       'tagline', 'overview', 'budget', 'revenue', 'origin_country',
       'original_language', 'spoken_languages', 'backdrop_path', 'poster_path',
       'production_companies', 'production_countries',
       'belongs_to_collection.name', 'id', 'imdb_id', 'imdbID',
       'genre_principal', 'production_companies_names', 'genre_1', 'genre_2',
       'genre_3', 'genre_4', 'genre_5', 'genre_6', 'genre_7'],
      dtype='str')>

In [176]:
df["Oscars_Gagnés"].value_counts()
df["Recompenses_globales"].describe()

count    4333.000000
mean        7.413339
std        18.091480
min         0.000000
25%         0.000000
50%         2.000000
75%         6.000000
max       245.000000
Name: Recompenses_globales, dtype: float64

In [177]:
# Colonnes à afficher à l'utilisateur 
colonnes_affichage = ["title", "overview", "poster_path", "Director", 
                      "Actors_1", "Actors_2", "Actors_3", "genre_principal"]

# Colonnes que le modèle va utiliser pour calculer la similarité
colonnes_modele = ["genre_1", "genre_2", "genre_3", "Year", "runtime", 
                   "vote_average"]

# On garde un seul df master avec tout, on séparera juste au moment d'entraîner
df_clean = df[colonnes_affichage + colonnes_modele].copy()
df_clean.shape

(4333, 14)

In [178]:
df_clean.head(3)

,title,overview,poster_path,Director,Actors_1,Actors_2,Actors_3,genre_principal,genre_1,genre_2,genre_3,Year,runtime,vote_average
0,Avatar,"Un marine paraplégique, envoyé sur la lune Pan...",/7N8L80OaG8EBTDdRBjGqT8PBzCM.jpg,James Cameron,Sam Worthington,Zoe Saldaña,Sigourney Weaver,Science-Fiction,Science-Fiction,Action,Aventure,2009,166.0,7.602
1,Pirates des Caraïbes : Jusqu'au bout du monde,L’âge d’or de la piraterie touche à sa fin. Mê...,/3JjceHUzbtgJ4fJs6x2ow4ckH89.jpg,Gore Verbinski,Johnny Depp,Orlando Bloom,Keira Knightley,Aventure,Aventure,Fantastique,Action,2007,168.0,7.266
2,Spectre,Un message cryptique surgi du passé entraîne J...,/AwCpcShhgRqoVPWhOzs4JiI4dlK.jpg,Sam Mendes,Daniel Craig,Christoph Waltz,Léa Seydoux,Action,Action,Aventure,Thriller,2015,148.0,6.569


In [179]:
df_clean.isnull().sum()

title                 0
overview            181
poster_path           1
Director              0
Actors_1              0
Actors_2              1
Actors_3              4
genre_principal       2
genre_1               2
genre_2             548
genre_3            1896
Year                  0
runtime               0
vote_average          0
dtype: int64

In [180]:
#on remplace les genres NA par "aucun genre"

df_clean["genre_2"]=df_clean["genre_2"].fillna("Aucun genre")
df_clean["genre_3"]=df_clean["genre_3"].fillna("Aucun genre")

In [181]:
#on remplace les données overview NA 
df_clean["overview"]=df_clean["overview"].fillna("Synopsis non disponible")

In [182]:
#on supprime les NA des colonnes poster_path(1), actors_2(1), actors(4), genre principal(4), genre_1(2)
df_clean=df_clean.dropna()

In [183]:
df_clean.isnull().sum()

title              0
overview           0
poster_path        0
Director           0
Actors_1           0
Actors_2           0
Actors_3           0
genre_principal    0
genre_1            0
genre_2            0
genre_3            0
Year               0
runtime            0
vote_average       0
dtype: int64

## preparation encodage ##


In [184]:
# assemblage des colonnes genre 1 à 3 car si on laisse tels quel le get dummie genre_1 action et genre_2_action 
# alors on auras des colonnes du même genres eclatés 

df_clean["Tous genres"]=df_clean[["genre_1","genre_2","genre_3"]].values.tolist()  #.values.tolist() >>> transforme action, aventure... en liste pour chaque films
df_clean["Tous genres"]
df_clean.isnull().sum()

title              0
overview           0
poster_path        0
Director           0
Actors_1           0
Actors_2           0
Actors_3           0
genre_principal    0
genre_1            0
genre_2            0
genre_3            0
Year               0
runtime            0
vote_average       0
Tous genres        0
dtype: int64

In [185]:
#on explose les colonnes Tous genres pour encodages// 
# déplie une colonne de listes en plusieurs lignes (l'index reste le même → c'est ce qui permet de regrouper après).
genres_explodes=df_clean["Tous genres"].explode()
genres_explodes.head(10)

0    Science-Fiction
0             Action
0           Aventure
1           Aventure
1        Fantastique
1             Action
2             Action
2           Aventure
2           Thriller
3             Action
Name: Tous genres, dtype: str

In [186]:
#on encode les colones avec get dummies
#.str.get_dummies() : l'accesseur .str dit à pandas "traite ça comme du texte".
# get_dummies() crée des colonnes 0/1 par valeur unique.
genre_dumies=genres_explodes.str.get_dummies()
genre_dumies.head(10)


,Action,Animation,Aucun genre,Aventure,Comédie,Crime,Documentaire,Drame,Familial,Fantastique,Guerre,Histoire,Horreur,Musique,Mystère,Romance,Science-Fiction,Thriller,Téléfilm,Western
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [187]:
#on regroupe les données par valeurs avec groupby 

#Regroupe les lignes par film (level=0 = index), 
# et pour chaque colonne (genre), garde 1 si au moins une ligne du film l'avait, sinon 0."

genres_encoded = genre_dumies.groupby(level=0).max()

print("Shape :", genres_encoded.shape)
print("Colonnes :", genres_encoded.columns.tolist())
genres_encoded.head(5)

Shape : (4326, 20)
Colonnes : ['Action', 'Animation', 'Aucun genre', 'Aventure', 'Comédie', 'Crime', 'Documentaire', 'Drame', 'Familial', 'Fantastique', 'Guerre', 'Histoire', 'Horreur', 'Musique', 'Mystère', 'Romance', 'Science-Fiction', 'Thriller', 'Téléfilm', 'Western']


,Action,Animation,Aucun genre,Aventure,Comédie,Crime,Documentaire,Drame,Familial,Fantastique,Guerre,Histoire,Horreur,Musique,Mystère,Romance,Science-Fiction,Thriller,Téléfilm,Western
0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


In [188]:
df_clean.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['title', 'overview', 'poster_path', 'Director', 'Actors_1', 'Actors_2',
       'Actors_3', 'genre_principal', 'genre_1', 'genre_2', 'genre_3', 'Year',
       'runtime', 'vote_average', 'Tous genres'],
      dtype='str')>

In [189]:
# Supprimer la colonne "Aucun" (elle n'apporte aucune info utile)
if "Aucun" in genres_encoded.columns:
    genres_encoded = genres_encoded.drop(columns=["Aucun"])

print("Shape après suppression de Aucun :", genres_encoded.shape)
print("Colonnes :", genres_encoded.columns.tolist())

Shape après suppression de Aucun : (4326, 20)
Colonnes : ['Action', 'Animation', 'Aucun genre', 'Aventure', 'Comédie', 'Crime', 'Documentaire', 'Drame', 'Familial', 'Fantastique', 'Guerre', 'Histoire', 'Horreur', 'Musique', 'Mystère', 'Romance', 'Science-Fiction', 'Thriller', 'Téléfilm', 'Western']


## determination de X ##

In [190]:
# # Réinitialiser les index pour garantir un alignement propre lors du concat.
# Après un dropna(), les index peuvent avoir des "trous" (ex: 0, 1, 2, 4, 6...).
# pd.concat(axis=1) aligne sur l'index : si les index ne matchent pas entre
# les 2 dataframes, on se retrouve avec des NaN. reset_index(drop=True) remet
# les index à 0, 1, 2... en continu, ce qui évite ce bug.

df_clean = df_clean.reset_index(drop=True)
genres_encoded = genres_encoded.reset_index(drop=True)


In [191]:
# Sélection des features numériques
X_numerique = df_clean[["Year", "runtime", "vote_average"]]

# Concaténation horizontale (axis=1 = côte à côte) avec les genres encodés
X = pd.concat([X_numerique, genres_encoded], axis=1)
X.shape

(4326, 23)

In [192]:
X=X.drop(columns="Aucun genre")

In [193]:
X.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['Year', 'runtime', 'vote_average', 'Action', 'Animation', 'Aventure',
       'Comédie', 'Crime', 'Documentaire', 'Drame', 'Familial', 'Fantastique',
       'Guerre', 'Histoire', 'Horreur', 'Musique', 'Mystère', 'Romance',
       'Science-Fiction', 'Thriller', 'Téléfilm', 'Western'],
      dtype='str')>

## standardisation avec standard Scaler ##

In [194]:
from sklearn.preprocessing import MinMaxScaler

# 1. Créer l'objet scaler
scaler = MinMaxScaler()

In [195]:
# 2. Sélectionner UNIQUEMENT les colonnes numériques à scaler
colonnes_a_scaler = ["Year", "runtime", "vote_average"]

In [196]:
# 3. Appliquer le scaling (fit_transform = apprend les min/max ET transforme)
X[colonnes_a_scaler] = scaler.fit_transform(X[colonnes_a_scaler])

#Le scaler fait 2 choses :

#.fit(X) → apprend les min et max de chaque colonne
#.transform(X) → applique la formule (valeur - min) / (max - min) pour ramener entre 0 et 1

In [197]:
# 4.  le résultat
X.head()

,Year,runtime,vote_average,Action,Animation,Aventure,Comédie,Crime,Documentaire,Drame,...,Guerre,Histoire,Horreur,Musique,Mystère,Romance,Science-Fiction,Thriller,Téléfilm,Western
0,0.921348,0.556962,0.871889,1,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0.898876,0.565401,0.833352,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.988764,0.481013,0.753412,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0.955056,0.548523,0.893795,1,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0.955056,0.447257,0.729327,1,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [198]:
#suppression doublon colonnes

X = X.loc[:, ~X.columns.duplicated()]
X.shape
X.columns.tolist()

['Year',
 'runtime',
 'vote_average',
 'Action',
 'Animation',
 'Aventure',
 'Comédie',
 'Crime',
 'Documentaire',
 'Drame',
 'Familial',
 'Fantastique',
 'Guerre',
 'Histoire',
 'Horreur',
 'Musique',
 'Mystère',
 'Romance',
 'Science-Fiction',
 'Thriller',
 'Téléfilm',
 'Western']

## entrainement du modele avec NearestNeighbors ##

recherche du plus proche voisin, non supervisé

In [199]:
from sklearn.neighbors import NearestNeighbors 

In [200]:
X.isnull().sum()

Year               0
runtime            0
vote_average       0
Action             0
Animation          0
Aventure           0
Comédie            0
Crime              0
Documentaire       0
Drame              0
Familial           0
Fantastique        0
Guerre             0
Histoire           0
Horreur            0
Musique            0
Mystère            0
Romance            0
Science-Fiction    0
Thriller           0
Téléfilm           0
Western            0
dtype: int64

In [206]:
# Créer le modèle (k=6 : on prendra 5 recos + le film lui-même)
modele = NearestNeighbors(n_neighbors=6)

# Entraîner sur X
modele.fit(X)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [212]:
# Choisis ton film de test
film_test = "Avatar"

# Trouver son index
idx = df_clean[df_clean["title"] == film_test].index[0]

# Demander les 6 plus proches voisins
distances, indices = modele.kneighbors(X.iloc[[idx]])

# Afficher les recommandations
print(f"Recommandations pour : {film_test}\n")
df_clean.iloc[indices[0]][["title", "Year", "genre_principal", "vote_average"]]

Recommandations pour : Avatar



,title,Year,genre_principal,vote_average
0,Avatar,2009,Science-Fiction,7.602
184,Hunger Games : L'Embrasement,2013,Aventure,7.426
779,Avengers,2012,Science-Fiction,8.010
16,Avengers,2012,Science-Fiction,8.010
26,Captain America : Civil War,2016,Aventure,7.447
432,Hunger Games,2012,Science-Fiction,7.222


## Sauvegarde ##

In [ ]:
import joblib

# 1. Sauvegarder df_clean (pour l'affichage)
df_clean.to_csv("films_clean.csv", index=False)

# 2. Sauvegarder le modèle entraîné 
#Un CSV ne peut pas stocker tout ça. joblib (et pickle) 
# sérialisent l'objet Python entier sur disque, et le restaurent à l'identique.
joblib.dump(modele, "modele.pkl")

# 3. Sauvegarder X (les features scalées)
X.to_csv("X_scaled.csv", index=False)

